In [ ]:
# Setup — all imports live here so the notebook executes top-to-bottom.
# When running headlessly (CI, HPC), uncomment the ``matplotlib.use('Agg')``
# line *before* the pyplot import so figures never need an interactive display.
import dataclasses
import logging
import os
import sys
from pathlib import Path

# import matplotlib
# matplotlib.use('Agg')  # enable for headless runs
import matplotlib.pyplot as plt
import mne
import numpy as np
import seaborn as sns
from matplotlib.lines import Line2D
from scipy.stats import pearsonr

# Resolve project root regardless of where the notebook is launched from
for _candidate in [".", "..", "../.."]:  # noqa: B007
    _p = os.path.abspath(_candidate)
    if os.path.isdir(os.path.join(_p, "src")):
        sys.path.insert(0, _p)
        break

from independent_vector_analysis import iva_g  # noqa: E402
from sklearn.decomposition import PCA  # noqa: E402

from scripts.notebook_helpers import (  # noqa: E402
    WAVELET_FREQ_MAX,
    WAVELET_FREQ_MIN,
    WAVELET_N_FREQS,
    analyzers_to_datasets,
    compute_wavelet_datasets,
    load_analyzers,
    participant_labels,
)
from src.analysis import iva_quality  # noqa: E402
from src.analysis.wavelet_ica import (  # noqa: E402
    align_iva_component_signs,
    iva_component_patterns,
    normalize_patterns_per_subject,
    zscore_by_time,
)
from src.definitions.constants import AssrEpoch, ProjectPaths  # noqa: E402
from src.definitions.fields import (  # noqa: E402
    ConditionVariants,
    ExclusionCategories,
    ExperimentNames,
    MusicTypeVariants,
)
from src.visualization.iva_quality_plots import (  # noqa: E402
    plot_full_tf_maps,
    plot_participant_tf_maps,
    plot_participant_topomaps,
    plot_tf_diagnostic,
    plot_topomap_diagnostic,
    plot_wavelet_reference,
)

logging.basicConfig(level=logging.INFO, format="%(levelname)s %(name)s: %(message)s")
sns.set_theme(style="whitegrid", palette="muted", font_scale=1.0)
mne.set_log_level("ERROR")
%matplotlib inline
print("Setup complete.")

# IVA Channel Decomposition — Quality via Topomap, Time & TF Correlations

This notebook **extends** the channel-as-independent IVA workflow
([`wavelet_iva_channel.ipynb`](wavelet_iva_channel.ipynb)) with a **quality
read-out** of the decomposition. It runs the same IVA-G pipeline (z-score →
per-subject channel PCA → IVA-G → sign alignment) to recover, per subject `s`
and per component `k`:

- a **channel topography** `iva_components[s, k]` — shape `(C,)`
- a **shared spectro-temporal source** `iva_sources[s, k]` — shape `(F, T)`

It then scores every `(subject, component)` pair against **one reference pair**,
and shows each y-axis on its own scatterplot.

---

## The reference pair (Step 3)

Both views come from **one** channel-PCA of the trial-averaged, time-z-scored
**wavelet power** — the reduction of
[`00-preprocessing/assr_wavelet_pca_analysis.ipynb`](../00-preprocessing/assr_wavelet_pca_analysis.ipynb)
and `scripts/run_assr_wavelet_pca.py`, applied to the IVA's own input tensor
`bb_z`:

- **the topography** — PC1's **channel loading**, `ref_topo`.
- **the TF map** — PC1's **score map** reshaped to `(F, W)`, `ref_tf`.

The loading and the score map carry the same per-subject PCA sign, so one
per-subject flip aligns both and the two views stay mutually consistent. The
*global* sign is then set by the physics: the pair is flipped until the driven
40 Hz response in its own TF map reads positive (Step 3), because the boxcar
y-axis below is an absolute reference and must not be read against an arbitrary
polarity.

> **Why not a raw-voltage reference?** An earlier version of this notebook also
> scored the patterns against the group PC1 of the raw **evoked voltage**
> response. That reference has been **removed**: the IVA decomposes wavelet
> *power*, so its channel patterns are power patterns. Power is a non-negative,
> slowly varying envelope, while an evoked voltage topography is a signed,
> phase-locked average of the raw signal whose polarity has no counterpart in
> power. Correlating the two does not answer "does this component reproduce the
> response" — only whether two unrelated quantities happen to co-vary across
> channels.

---

## The axes

**x — topomap correlation (Step 4).** The component's forward channel pattern
against `ref_topo`. Both are `(C,)` in the same channel order, so this is
pattern-vs-pattern.

**y — onset-locked, rigid 500 ms boxcar (Steps 2 and 5).** The ASSR is delivered
as **continuous 40 Hz stimulation**, so a *sustained* response does not vary in
time and is invisible under `zscore_by_time` (which is why an on/off indicator
over the whole recording gave correlations ≈ 0). What *is* time-locked is the
**transient right after each `fam+` onset**, so the reference is built around the
onset:

1. **Onset-lock & average.** Reduce each IVA source `(F, T)` to a `(T,)` time
   course two ways, then epoch it around every onset (`[-EPOCH_PRE_S,
   EPOCH_POST_S]`) and average → the component's onset-triggered mean response
   with a pre-onset baseline.
2. **Compare against one rigid boxcar.** The "expected response" is `0` before
   the onset, `1` for the fixed `RESP_DURATION_S` = **500 ms** stimulus window
   after it, and `0` afterwards. The window is **not** fitted per component, so
   every component is scored against the same reference. Zero-lag Pearson.

   The two frequency reductions: the **PCA variant** (first PC over frequency)
   and the **40 Hz variant** (the single `40 Hz` row, the stimulation frequency).

**y — TF-map correlation (Step 5).** The component's *onset-averaged* `(F, W)`
map against `ref_tf`, as one Pearson correlation over the flattened map.
Frequency is never collapsed, so a component with the right time course at the
wrong frequency scores low. Also computed **per frequency band**
(`iva_quality.TF_BANDS`): **1-10 Hz** (the onset transient and slow evoked shape)
and **30-50 Hz** (the band around the steady state), which ask the same question
where the response is expected instead of over the whole spectrum.

---

## Output

Five scatterplots, each point a `(subject, component)` pair, **colour =
component**. Every score carries one shared per-component sign, so the one x may
be read against any y — which is what makes the grid below interpretable:

| # | y (temporal reference) | Step |
|---|---|---|
| 1 | boxcar, PCA freq-reduction | 6 |
| 2 | boxcar, 40 Hz only | 6 |
| 3 | onset-averaged TF map, whole | 6b |
| 4 | onset-averaged TF map, 1-10 Hz | 6b |
| 5 | onset-averaged TF map, 30-50 Hz | 6b |

Read them in pairs, changing one axis at a time. The x never moves, so a
component that climbs from **1** to **3** matches the reference's whole
time-frequency structure better than the paradigm's window, and one that climbs
from **3** to **5** is the ASSR and little else — diluted, in the whole-map
score, by every frequency the reference is quiet in.

> **Signs.** Both PCA and IVA fix component signs only up to a flip. Left
> unhandled, this would turn genuine matches into negative correlations. **Step
> 5b** controls the flip *per component*: IVA's alignment makes each component's
> sign consistent across subjects, the frequency-PCA's extra per-subject flip is
> anchored to the signed source, and each component is finally oriented so its
> **topomap** correlation reads positive — a genuine topomap-vs-y sign
> disagreement is preserved, and `|r|` is invariant to all of it. **Step 5c**
> then resolves the flip *per participant* for every figure that compares or
> averages participants, because IVA leaves each `(subject, component)` pair's
> polarity free and mixed polarities cancel in a group mean. That is **one**
> sign per pair — taken from its topography against `ref_topo` and worn by its
> TF map too, a pattern and its source being one decomposition. **Step 7c**
> then draws the whole recording under those same signs, as the QC counterpart
> of the standard IVA time-frequency figure.

In [ ]:
# ── Experiment configuration ───────────────────────────────────
# The reference topomap is defined from the ASSR stimulus-locked evoked
# response, so this quality analysis targets the ASSR experiment.
EXPERIMENT_NAME = ExperimentNames.ASSR
CONDITION = ConditionVariants.PLACEBO  # default per project convention
if EXPERIMENT_NAME == ExperimentNames.ASSR:
    MUSIC_TYPES = [MusicTypeVariants.ASSR]
else:
    MUSIC_TYPES = [MusicTypeVariants.CLASSICAL]
EXCLUSION_CATEGORIES = [ExclusionCategories.BAD_MUSIC, ExclusionCategories.ARTIFACTS]
PROCESS_AND_SAVE_DATA = False  # set True to re-process raw files

# ── Wavelet settings ─────────────────────────────────────────
REPRESENTATION = "power"
FREQS = np.linspace(WAVELET_FREQ_MIN, WAVELET_FREQ_MAX, WAVELET_N_FREQS)
KEEP_FREQUENCY_DIM = True
RESHAPE_FREQUENCY_DIM = True  # -> (n_subjects, n_channels, n_freqs, n_times)
REUSE_WAVELETS = True  # load from cache; set False to compute + save

# ── Subject / channel / time subset (mirror wavelet_iva_channel.ipynb) ──
N_SUBJECTS_SUBSET: int | None = 5
N_CHANNELS_SUBSET: int | None = 32   # first N channels (PCA reduces this axis)
N_TIMES_SUBSET: int | None = 3000    # first N time samples (F*T = n_freqs * this)

# ── IVA settings ──────────────────────────────────────────────
# 10 components is the standard ASSR setting for this variant and the
# default of scripts/run_wavelet_iva_channel.py: the question is whether a
# handful of components reproduce the stimulus response, and every
# per-component figure stays readable at that size.
N_COMPONENTS_PCA = 10  # per-subject PCA dim over channels (= N in IVA's (N, T, K))
IVA_OPT_APPROACH = "newton"
IVA_MAX_ITER = 64
IVA_W_DIFF_STOP = 1e-6
IVA_VERBOSE = False
IVA_RANDOM_STATE = 42

# ── Stimulus-locked epoch geometry (shared by BOTH quality references) ──────
# Taken from the paradigm definition (src.definitions.constants.AssrEpoch), so
# this notebook, the headless `run_wavelet_iva_channel.py --quality` path
# (src.analysis.iva_quality) and the 00-preprocessing ASSR PCA notebooks all cut
# the SAME window:
#   0.1 s baseline before onset, then 1.0 s after onset
#   = 0.5 s stimulus + 0.5 s post-stimulus
# The post-onset length is capped by the shortest inter-onset gap wherever it is
# applied, so no epoch can reach a neighbouring stimulus.
EPOCH_PRE_S = AssrEpoch.PRE_ONSET_S    # pre-onset baseline (both references)
EPOCH_POST_S = AssrEpoch.POST_ONSET_S  # post-onset span (both references)

# The rigid expected-response window for the boxcar reference is the stimulus
# length itself — the OFF edge is the stimulus offset, so it is the same number.
RESP_DURATION_S = AssrEpoch.STIMULUS_DURATION_S

# Reference channel that anchors the group polarity of the PC1 loading.
REF_POLARITY_CHANNEL = iva_quality.REF_POLARITY_CHANNEL
# Frequency (Hz) used by the "40 Hz-only" time-reduction variant.
ASSR_FREQ = iva_quality.ASSR_FREQ

# ── Which components to score ─────────────────────────────────────────
# None = every IVA component (0 .. N_COMPONENTS_PCA-1). Set to an explicit list
# of 0-based component indices to focus the scatterplot on a subset.
COMPONENTS_TO_PLOT: list[int] | None = None

# ── Plot saving ───────────────────────────────────────────────────────
SAVE_PLOTS = True
PLOTS_DIR = (
    ProjectPaths.NOTEBOOKS_DIR
    / "05-wavelet-iva-analysis"
    / "plots"
    / EXPERIMENT_NAME.value
    / "broadband"
    / "iva_channel_quality"
    / f"pca_{N_COMPONENTS_PCA}"
)
PLOTS_DIR.mkdir(parents=True, exist_ok=True)

print(f"Experiment / group : {EXPERIMENT_NAME.value} — {CONDITION.value}")
print(f"Frequencies        : {FREQS[0]:.1f}-{FREQS[-1]:.1f} Hz ({len(FREQS)} steps)")
print(f"Per-subject PCA dim : {N_COMPONENTS_PCA}  (over channels)")
print(f"Onset epoch        : [-{EPOCH_PRE_S}, {EPOCH_POST_S}] s around each onset "
      f"({AssrEpoch.STIMULUS_DURATION_S} s stimulus + "
      f"{AssrEpoch.POST_STIMULUS_S} s post-stimulus)")
print(f"Response window    : {RESP_DURATION_S * 1000:.0f} ms (rigid, onset-locked)")
print(f"Plots -> {PLOTS_DIR}")

In [ ]:
analyzers = load_analyzers(
    MUSIC_TYPES,
    CONDITION,
    EXCLUSION_CATEGORIES,
    PROCESS_AND_SAVE_DATA,
    normalize_data=False,
    experiment_name=EXPERIMENT_NAME,
)
datasets = analyzers_to_datasets(analyzers)

# Subset subjects / channels / times (applied to ``datasets`` only — the full
# ``analyzers[...].data`` is kept intact and reused for the reference topomap).
if N_SUBJECTS_SUBSET is not None:
    datasets = {
        label: dataclasses.replace(ad, data=ad.data[:N_SUBJECTS_SUBSET])
        for label, ad in datasets.items()
    }
if N_CHANNELS_SUBSET is not None:
    datasets = {
        label: dataclasses.replace(ad, data=ad.data[:, :N_CHANNELS_SUBSET, :])
        for label, ad in datasets.items()
    }
if N_TIMES_SUBSET is not None:
    datasets = {
        label: dataclasses.replace(ad, data=ad.data[:, :, :N_TIMES_SUBSET])
        for label, ad in datasets.items()
    }

print("Loaded datasets:", list(datasets.keys()))
for label, ad in datasets.items():
    print(
        f"  {label}: {ad.n_items} subjects, {ad.n_features} channels, "
        f"{ad.n_samples} samples"
    )

In [ ]:
broadband_datasets = compute_wavelet_datasets(
    datasets=datasets,
    analyzers=analyzers,
    experiment_name=EXPERIMENT_NAME,
    freqs=FREQS,
    representation=REPRESENTATION,
    keep_frequency_dim=KEEP_FREQUENCY_DIM,
    reshape_frequency_dim=RESHAPE_FREQUENCY_DIM,
    wavelet_dir=(
        ProjectPaths.NOTEBOOKS_DIR
        / "03-wavelet-analysis"
        / "wavelet_cache"
        / EXPERIMENT_NAME.value
        / "broadband"
    ),
    reuse_wavelets=REUSE_WAVELETS,
)
for label, ad in broadband_datasets.items():
    source = ad.metadata.get("loaded_from_wavelet_file", "computed_now")
    print(f"[broadband] {label}: shape={ad.data.shape}  source={source}")

In [ ]:
LABEL = list(broadband_datasets.keys())[0]

bb_ad = broadband_datasets[LABEL]
bb_data = bb_ad.data  # (n_subjects, n_channels, n_freqs, n_times)
sfreq = bb_ad.sfreq

n_subjects, n_channels, n_freqs, n_times = bb_data.shape
time = np.arange(n_times) / sfreq

# MNE info for the topomaps, restricted to the IVA channel subset. This is the
# canonical channel order of ``iva_components`` below.
iva_info = analyzers[LABEL].info
iva_info = mne.pick_info(iva_info, mne.pick_types(iva_info, eeg=True))
if n_channels < len(iva_info.ch_names):
    iva_info = mne.pick_info(iva_info, list(range(n_channels)))
iva_ch_names = list(iva_info["ch_names"])

print(f"Dataset    : {LABEL}")
print(f"Shape      : {bb_data.shape}  (subjects x channels x freqs x times)")
print(f"Duration   : {n_times / sfreq:.1f} s  @  {sfreq} Hz")
print(f"Freq range : {FREQS[0]:.1f}-{FREQS[-1]:.1f} Hz ({n_freqs} steps)")
print(f"IVA channels : {n_channels}  (first={iva_ch_names[0]}, last={iva_ch_names[-1]})")

# Per-subject participant labels (``019`` etc.) taken from the metadata
# sidecar written next to the concatenated array. ``participant_labels`` maps
# subject index -> CONCATENATED_PERSON_INDEX -> PARTICIPANT_ID and raises if the
# mapping is unavailable, so a wrong label can never pass silently.
try:
    subject_ids = participant_labels(analyzers[LABEL].filtered_df, n_subjects)
except ValueError as _exc:
    print(f"Participant labels unavailable ({_exc}); using subject indices.")
    subject_ids = [f"#{s:03d}" for s in range(n_subjects)]

print(f"Subject IDs  : {subject_ids}")

## Step 1 — Run the channel-as-independent IVA-G

Identical to [`wavelet_iva_channel.ipynb`](wavelet_iva_channel.ipynb): z-score
along time, per-subject **PCA over channels** (square-mixing requirement of
IVA-G), run IVA-G, resolve the per-subject sign ambiguity, then recover the
per-subject spectro-temporal **sources** `(S, N_PCA, F, T)` and channel
**topographies** `(S, N_PCA, C)`.

In [ ]:
# Step 1a — z-score along time, per-subject reshape to (S, C, F*T).
bb_z = zscore_by_time(bb_data)
n_samples_ft = n_freqs * n_times
X_subjects = bb_z.reshape(n_subjects, n_channels, n_samples_ft)

if N_COMPONENTS_PCA > n_channels:
    raise ValueError(
        f"N_COMPONENTS_PCA ({N_COMPONENTS_PCA}) must be <= n_channels "
        f"({n_channels}); PCA reduces the channel axis here."
    )

# Step 1b — per-subject PCA over channels -> IVA input (N_PCA, F*T, S).
pcas: list[PCA] = []
pca_scores_per_subject = np.zeros((n_subjects, N_COMPONENTS_PCA, n_samples_ft))
for k in range(n_subjects):
    subj_matrix = X_subjects[k].T  # (F*T, C)
    pca = PCA(n_components=N_COMPONENTS_PCA, random_state=IVA_RANDOM_STATE)
    scores = pca.fit_transform(subj_matrix)
    pcas.append(pca)
    pca_scores_per_subject[k] = scores.T
X_pca = np.ascontiguousarray(pca_scores_per_subject.transpose(1, 2, 0))

# Step 1c — IVA-G.
rng = np.random.default_rng(IVA_RANDOM_STATE)
W_init = rng.standard_normal((N_COMPONENTS_PCA, N_COMPONENTS_PCA, n_subjects))
W, cost, Sigma_N, _isi = iva_g(
    X_pca,
    opt_approach=IVA_OPT_APPROACH,
    whiten=True,
    verbose=IVA_VERBOSE,
    W_init=W_init,
    max_iter=IVA_MAX_ITER,
    W_diff_stop=IVA_W_DIFF_STOP,
)

# Step 1d — resolve per-subject sign ambiguity (aligns W across subjects).
sigma_corr, W, sign_flips = align_iva_component_signs(Sigma_N, W)

# Step 1e — recover spectro-temporal sources + channel topographies.
iva_scores_pca = np.zeros((n_subjects, N_COMPONENTS_PCA, n_samples_ft))
iva_components = np.zeros((n_subjects, N_COMPONENTS_PCA, n_channels))
for k in range(n_subjects):
    W_k = W[:, :, k]
    iva_scores_pca[k] = W_k @ X_pca[:, :, k]
    # Forward (mixing) patterns — the reference PC1 channel loading is a
    # pattern too, so the correlation below is pattern-vs-pattern.
    iva_components[k] = iva_component_patterns(
        W_k, pcas[k].components_
    )  # (N_PCA, C)
iva_sources = iva_scores_pca.reshape(n_subjects, N_COMPONENTS_PCA, n_freqs, n_times)

print(f"IVA-G iterations   : {len(cost)}  final cost = {cost[-1]:.6f}")
print(f"iva_sources        : {iva_sources.shape}  (S, N_PCA, F, T)")
print(f"iva_components      : {iva_components.shape}  (S, N_PCA, C)")

## Step 2 — Onset-locked epoch window & the rigid boxcar reference

Set up the onset-triggered epoch and the single "expected response" boxcar.

The epoch is the **paradigm window**, obtained from `iva_quality.onset_window`
(which reads `src.definitions.constants.AssrEpoch`): `0.1` s baseline before
onset, then `1.0` s after onset — the `0.5` s **stimulus** plus `0.5` s of
**post-stimulus** — capped by the shortest inter-onset gap so epochs never
overlap. Step 3's reference pair is cut on the identical window, and so is the
headless `run_wavelet_iva_channel.py --quality` path, so no quality metric is
scored on a different epoch than another.

The boxcar is `0` before the onset, `1` for the rigid `RESP_DURATION_S`
(500 ms = the stimulus length, so the OFF edge is the stimulus offset), and `0`
afterwards. Keeping `0.5` s of epoch *after* that edge is what lets the score
distinguish a response that stops with the stimulus from one that runs on: with
a window that ended at the stimulus offset, both would look identical. The
`0`-tail is scored, not padding.

The window is clamped to the epoch's post-onset span so it always fits, and the
cell warns if the clamp ever engages. It is used in Step 5 to score every
component against one and the same reference.

In [ ]:
_onset_samples = analyzers[LABEL].stimulus_onsets
onsets_in = _onset_samples[_onset_samples < n_times].astype(int)

# Same paradigm window as the reference topomap (Step 2) and the headless path:
# 0.1 s baseline + 1.0 s post-onset, capped by the shortest inter-onset gap.
PRE, POST = iva_quality.onset_window(onsets_in, n_times, sfreq)
W = PRE + POST
epoch_times = np.arange(-PRE, POST) / sfreq  # (W,) seconds, t=0 at onset
stim_mask = AssrEpoch.stimulus_mask(epoch_times)  # the driven 0-0.5 s interval


def onset_average(x):
    """Onset-triggered average of a 1-D time course x(t) -> (W,)."""
    acc, n_used = None, 0
    for o in onsets_in:
        s, e = o - PRE, o + POST
        if s < 0 or e > n_times:
            continue
        seg = x[s:e]
        acc = seg.astype(float) if acc is None else acc + seg
        n_used += 1
    if n_used == 0:
        raise ValueError("No onset window fits inside the IVA time window.")
    return acc / n_used, n_used


def boxcar(d_samples):
    """Expected response over the epoch: 0 baseline, 1 for d post-onset samples."""
    b = np.zeros(W)
    b[PRE:PRE + int(d_samples)] = 1.0
    return b


# The rigid reference window: the stimulus length, clamped to the epoch's
# post-onset span so it always fits inside the epoch.
RESP_SAMPLES = iva_quality.response_duration_samples(POST, sfreq)
RESP_DUR_S = RESP_SAMPLES / sfreq
REF_BOXCAR = boxcar(RESP_SAMPLES)

print(f"Onsets used      : {len(onsets_in)}")
print(f"Epoch window     : {W} samples ({PRE} pre, {POST} post) = "
      f"[{epoch_times[0]:.3f}, {epoch_times[-1]:.3f}] s")
print(f"Response window  : {RESP_DUR_S * 1000:.0f} ms ({RESP_SAMPLES} samples), rigid")
print(f"Post-stimulus    : {epoch_times[-1] - RESP_DUR_S:.3f} s after the OFF edge")
if POST < int(round(EPOCH_POST_S * sfreq)):
    print(f"  NOTE: post-onset span trimmed from {EPOCH_POST_S} s to "
          f"{POST / sfreq:.3f} s by the shortest inter-onset gap.")
if RESP_SAMPLES < int(round(RESP_DURATION_S * sfreq)):
    print(f"  WARNING: the {RESP_DURATION_S * 1000:.0f} ms stimulus window does not "
          f"fit the epoch and was clamped to {RESP_DUR_S * 1000:.0f} ms.")

# Show the reference boxcar on the epoch time axis.
fig, ax = plt.subplots(figsize=(10, 2.4))
ax.plot(epoch_times, REF_BOXCAR, lw=1.6, drawstyle="steps-post", color="C0")
ax.axvline(0.0, color="red", ls="--", lw=0.8)
ax.set_ylim(-0.1, 1.1)
ax.set_yticks([0, 1])
ax.set_xlabel("Time relative to onset (s)")
ax.set_title(
    f"Rigid {RESP_DUR_S * 1000:.0f} ms onset-locked boxcar reference "
    f"(epoch [{epoch_times[0]:.1f}, {epoch_times[-1]:.1f}] s)"
)
fig.tight_layout()
if SAVE_PLOTS:
    fig.savefig(PLOTS_DIR / "boxcar_reference.png", dpi=150, bbox_inches="tight")
plt.show()
plt.close("all")

## Step 3 — The reference pair (channel-PCA of the trial-averaged wavelets)

**One** reference, two views of it, from **one** channel-PCA — the reduction of
[`00-preprocessing/assr_wavelet_pca_analysis.ipynb`](../00-preprocessing/assr_wavelet_pca_analysis.ipynb),
run here on the IVA's own input tensor `bb_z` so the reference and the component
maps are bin-for-bin comparable:

1. **z-score along time** — already done: `bb_z = zscore_by_time(bb_data)`
   normalises each `(subject, channel, frequency)` series, which is what removes
   the 1/f tilt so the PCA is not dominated by absolute low-frequency power.
2. **trial-average** around every onset over the Step 2 epoch → `(C, F, W)`.
3. **PCA over channels** — each `(freq, time)` bin an observation, each channel a
   variable. PC1's **channel loading** is the reference topography (`ref_topo`);
   PC1's **score map**, reshaped to `(F, W)`, is the reference TF map (`ref_tf`).
4. **polarity-align across subjects**, then average. Loading and score map carry
   the same arbitrary PCA sign, so one per-subject sign aligns both and the two
   views stay mutually consistent.

**Why the reference is a wavelet one.** The IVA decomposes wavelet *power*, so its
channel patterns are power patterns. Power is a non-negative, slowly varying
envelope; an evoked *voltage* topography is a signed, phase-locked average of the
raw signal, and its polarity has no counterpart in power. Correlating a
power-derived pattern with a voltage-derived topography therefore does not answer
"does this component reproduce the response" — it only asks whether two unrelated
quantities happen to co-vary across channels. The earlier raw-PCA reference has
been removed for that reason; every score below is against this pair.

**Global polarity.** The pair's overall sign is a PCA sign, i.e. arbitrary, and it
propagates into every score through the per-component orientation in Step 5b. The
boxcar y-axis (Step 3) is *not* arbitrary — it is positive during the stimulus —
so the pair is flipped until the **driven response in its own TF map reads
positive** (35-45 Hz inside the stimulus window, the same rule that aligns the
participants in Step 5c). Both halves flip together, so they stay consistent, and
`iva_quality.compute_iva_quality` does exactly this in the headless path.

All of this is `iva_quality.wavelet_reference`, the same function the headless
`run_wavelet_iva_channel.py --quality` path calls.

In [ ]:
ref_topo, ref_tf, ref_anchor, ref_consistency = iva_quality.wavelet_reference(
    bb_z,          # (S, C, F, T) already z-scored along time in Step 1a
    onsets_in,
    PRE,
    POST,
    iva_ch_names,
    polarity_channel=REF_POLARITY_CHANNEL,
)

# Global polarity: make the reference's own driven response positive, so the
# boxcar y-axis (an absolute reference) is not read against an arbitrary sign.
_ref_sign, _have_ref_anchor = iva_quality.anchor_signs_tf(
    ref_tf[None, None], FREQS, epoch_times, resp_duration_s=RESP_DUR_S
)
if _have_ref_anchor and _ref_sign[0, 0] < 0:
    ref_topo, ref_tf = -ref_topo, -ref_tf
    print("Reference pair flipped so its driven 40 Hz response reads positive.")

print(f"ref_topo : {ref_topo.shape}  (PC1 channel loading)")
print(f"ref_tf   : {ref_tf.shape}  (F, W) PC1 score map")
print(f"Polarity anchor channel : {ref_anchor} "
      f"(global sign then set by the driven response)")
print(f"Topographies agreeing   : {ref_consistency.n_agreeing}/"
      f"{ref_consistency.n_subjects}  "
      f"(median pairwise r={ref_consistency.median_pairwise_r:+.2f}, "
      f"weakest r={ref_consistency.min_subject_r:+.2f})")
if ref_consistency.n_agreeing < ref_consistency.n_subjects:
    print(f"  WARNING: "
          f"{ref_consistency.n_subjects - ref_consistency.n_agreeing} subject(s) "
          f"anti-correlate with the group PC1 topography — a topographic "
          f"outlier, not a sign problem. The reference averages them in.")

# Visual check: the two views of the same reference side by side.
fig = plot_wavelet_reference(
    ref_topo,
    ref_tf,
    iva_info,
    n_channels,
    FREQS,
    epoch_times,
    label=LABEL,
    resp_duration_s=RESP_DUR_S,
    assr_freq=ASSR_FREQ,
    consistency=ref_consistency,
    save_path=(PLOTS_DIR / "reference_wavelet.png") if SAVE_PLOTS else None,
)
plt.show()
plt.close("all")

## Step 4 — Topomap correlations (the one x-axis)

Pearson correlation between each `(subject, component)` topography
`iva_components[s, k]` and the reference topography `ref_topo`. Both are `(C,)` on
the same channel order, so this is pattern-vs-pattern. Signed — see the sign
caveat in the header and Steps 5b / 5c.

In [ ]:
topo_corr = np.zeros((n_subjects, N_COMPONENTS_PCA))
for s in range(n_subjects):
    for k in range(N_COMPONENTS_PCA):
        topo_corr[s, k] = pearsonr(iva_components[s, k], ref_topo)[0]

print(f"topo_corr : {topo_corr.shape}  (subjects x components)  "
      f"range [{topo_corr.min():+.2f}, {topo_corr.max():+.2f}]  "
      f"max |r| = {np.abs(topo_corr).max():.2f}")

## Step 5 — Onset-locked time & TF correlations (both y-axes)

**The boxcar y.** For each `(subject, component)` reduce the source `(F, T)`
to a `(T,)` time course two ways, **onset-average** it (Step 2 helper), then
correlate that averaged response against the **rigid boxcar** `REF_BOXCAR`
(zero-lag Pearson, the same 500 ms reference for every subject and component):

- **PCA variant** (`time_corr_pca`) — first PC over frequency (sign-anchored to
  the signed temporal marginal so it is consistent across subjects).
- **40 Hz variant** (`time_corr_40`) — the single `ASSR_FREQ` frequency row.

**The TF-map y.** No frequency reduction at all: epoch-average the source over
the *whole* `(F, T)` plane → `onset_tf[s, k]` of shape `(F, W)`, then correlate
that map against `ref_tf` as one flattened Pearson (`tf_corr`). Because both
axes of the map are scored together, a component whose time course looks right
but sits at the wrong frequency cannot score highly — which the frequency-reduced
variants above cannot distinguish.

**The TF-map y, band-limited.** The same score over one frequency band at a time
(`iva_quality.TF_BANDS`, applied through `tf_map_correlation`'s `freq_mask`):

- **1-10 Hz** (`tf_corr_1_10hz`) — the onset transient and the slow evoked shape.
- **30-50 Hz** (`tf_corr_30_50hz`) — the band bracketing the 40 Hz steady state,
  wide enough to survive wavelet smearing and a few Hz of stimulator drift.

The whole-map score asks whether a component matches the reference *anywhere* in
the spectrum; these ask it *where the response is expected*. A component that is
the ASSR and little else scores high on 30-50 Hz while the whole-map score dilutes
it with every frequency the reference is quiet in — so read the three together.

`onset_avg_pca` / `onset_avg_40` / `onset_tf` keep the per-subject maps for the
diagnostic plots.

In [ ]:
f40 = int(np.argmin(np.abs(FREQS - ASSR_FREQ)))
print(f"40 Hz row : FREQS[{f40}] = {FREQS[f40]:.1f} Hz")

# Onset-triggered average of each component's reduced time course, per variant.
onset_avg_pca = np.zeros((n_subjects, N_COMPONENTS_PCA, W))
onset_avg_40 = np.zeros((n_subjects, N_COMPONENTS_PCA, W))
for s in range(n_subjects):
    for k in range(N_COMPONENTS_PCA):
        src = iva_sources[s, k]  # (F, T)
        # Variant A — first PC over frequency, sign-anchored to the signed
        # temporal marginal (the frequency-PCA's own sign is otherwise arbitrary).
        pc1_time = PCA(n_components=1, random_state=IVA_RANDOM_STATE).fit_transform(
            src.T
        )[:, 0]
        if pearsonr(pc1_time, src.mean(axis=0))[0] < 0:
            pc1_time = -pc1_time
        onset_avg_pca[s, k], _ = onset_average(pc1_time)
        # Variant B — the single 40 Hz row (sign-aligned by IVA).
        onset_avg_40[s, k], _ = onset_average(src[f40])


def boxcar_correlation(onset_avgs):
    """Zero-lag Pearson of every onset-average against the rigid boxcar.

    ``onset_avgs`` is (S, K, W). ``REF_BOXCAR`` is the same reference for every
    subject and component, so the returned (S, K) scores are directly
    comparable across components.
    """
    n_comp = onset_avgs.shape[1]
    time_corr = np.zeros((n_subjects, n_comp))
    for k in range(n_comp):
        time_corr[:, k] = np.nan_to_num(
            np.array(
                [
                    pearsonr(onset_avgs[s, k], REF_BOXCAR)[0]
                    for s in range(n_subjects)
                ]
            )
        )
    return time_corr


time_corr_pca = boxcar_correlation(onset_avg_pca)
time_corr_40 = boxcar_correlation(onset_avg_40)

# The TF-map y — no frequency reduction: epoch-average the whole (F, T) plane in one
# call (``epoch_average`` averages along the last axis and preserves the leading
# ones), then score each (F, W) map against ref_tf as a single flattened Pearson.
onset_tf, _n_used_tf = iva_quality.epoch_average(iva_sources, onsets_in, PRE, POST)
tf_corr = iva_quality.tf_map_correlation(onset_tf, ref_tf)

# The same score restricted to one frequency band at a time. A band with no bin
# inside FREQS is skipped rather than zero-filled, so `in` is the availability
# check downstream (as `have_40hz` is for the 40 Hz time variant).
tf_corr_bands = {}
for _band_name, _fmin, _fmax, _tag in iva_quality.TF_BANDS:
    _mask = iva_quality.frequency_band_mask(FREQS, _fmin, _fmax)
    if _mask.any():
        tf_corr_bands[_tag] = iva_quality.tf_map_correlation(
            onset_tf, ref_tf, freq_mask=_mask
        )
    else:
        print(f"{_band_name} outside [{FREQS.min():.1f}, {FREQS.max():.1f}] Hz "
              f"-> band TF variant skipped.")

for name, arr in (
    ("PCA-reduced", time_corr_pca),
    ("40 Hz", time_corr_40),
):
    print(f"time_corr [{name:11s}] range [{arr.min():+.2f}, {arr.max():+.2f}]  "
          f"max |r| = {np.abs(arr).max():.2f}")
print(f"onset_tf  : {onset_tf.shape}  (S, K, F, W) from {_n_used_tf} onsets")
print(f"tf_corr   : range [{tf_corr.min():+.2f}, {tf_corr.max():+.2f}]  "
      f"max |r| = {np.abs(tf_corr).max():.2f}")
for _band_name, _fmin, _fmax, _tag in iva_quality.TF_BANDS:
    if _tag in tf_corr_bands:
        _arr = tf_corr_bands[_tag]
        # The bins actually covered: a band the wavelet range only partly reaches
        # is scored over what it has, so print it rather than trust the label.
        _band_freqs = FREQS[iva_quality.frequency_band_mask(FREQS, _fmin, _fmax)]
        print(f"tf_corr [{_band_name:8s}] range [{_arr.min():+.2f}, {_arr.max():+.2f}]"
              f"  max |r| = {np.abs(_arr).max():.2f}  ({len(_band_freqs)} bins, "
              f"{_band_freqs.min():.1f}-{_band_freqs.max():.1f} Hz)")

## Step 5b — Sign-ambiguity check & per-component orientation

IVA and PCA fix component signs only up to a flip, which could turn a genuine
match into a **negative** correlation. Two flips are in play, and both are now
controlled:

- **Cross-subject flip** — `align_iva_component_signs` already oriented every
  component's sign consistently across subjects, so a component's **topography**
  and its **40 Hz source** point the same way for all subjects. Because a
  component's topography and source share one sign, the topomap, both time
  correlations and the TF correlation flip *together*.
- **Frequency-PCA flip** — the PCA-reduced time course carried an extra
  per-subject sign, anchored above to the signed temporal marginal. (The TF
  variant has no such extra flip: nothing is re-decomposed, the source's own
  `(F, T)` plane is epoch-averaged as it stands.)

This cell **verifies** cross-subject sign consistency (how many subjects agree
with each component's mean-sign — low agreement means the pattern genuinely
differs across subjects, *not* a sign artefact), then gives each component one
**global orientation** so its **topomap** correlation reads positive. Every
other measure follows from that one shared component sign — the TF score is *not*
oriented on its own: the topography and the TF map are two views of one reference
and are already mutually aligned (Step 3), so a TF score still reading negative is
a genuine spatial-vs-temporal disagreement rather than a sign artefact. `|r|` is
unchanged by any of this.

In [ ]:
def _sign_agreement(corr):
    """Per-component fraction of subjects agreeing with the mean-sign. (S,K)->(K,)."""
    frac = np.zeros(corr.shape[1])
    for k in range(corr.shape[1]):
        col = corr[:, k]
        ref = np.sign(col.mean()) or 1.0
        frac[k] = float(np.mean(np.sign(col) == ref))
    return frac


for name, arr in [
    ("topomap", topo_corr),
    ("time-PCA", time_corr_pca),
    ("time-40Hz", time_corr_40),
    ("TF-map", tf_corr),
] + [(f"TF {tag}", arr) for tag, arr in tf_corr_bands.items()]:
    frac = _sign_agreement(arr)
    weak = [k + 1 for k in range(N_COMPONENTS_PCA) if frac[k] < 0.6]
    print(
        f"[{name:9s}] cross-subject sign agreement: mean {frac.mean() * 100:3.0f}%, "
        f"min {frac.min() * 100:3.0f}%  |  <60% agreement: {weak or 'none'}"
    )

# Global per-component orientation: one flip per component so its mean topomap
# correlation is >= 0. The same flip applies to every measure (shared component
# sign), so it is applied to every correlation array. Idempotent.
sign_per_comp = np.sign(topo_corr.mean(axis=0))
sign_per_comp[sign_per_comp == 0] = 1.0
topo_corr = topo_corr * sign_per_comp
time_corr_pca = time_corr_pca * sign_per_comp
time_corr_40 = time_corr_40 * sign_per_comp
tf_corr = tf_corr * sign_per_comp
# The band-limited TF scores are y-axes of their own, so they follow the same
# shared component sign as the whole-map one.
tf_corr_bands = {tag: arr * sign_per_comp for tag, arr in tf_corr_bands.items()}
# Orient the onset-averages and the TF maps the same way so the diagnostic plots
# match the oriented correlations (a positive-going response for a positive score).
onset_avg_pca = onset_avg_pca * sign_per_comp[None, :, None]
onset_avg_40 = onset_avg_40 * sign_per_comp[None, :, None]
onset_tf = onset_tf * sign_per_comp[None, :, None, None]
n_flipped_comp = int((sign_per_comp < 0).sum())
print(
    f"Oriented {n_flipped_comp}/{N_COMPONENTS_PCA} components so the mean topomap "
    f"correlation is >= 0 (every other sign follows the shared component sign)."
)

## Step 5c — Per-participant sign anchoring (for every map comparison)

Step 5b resolved **one** sign per component, which is what the scores need. It
does *not* make the participants agree with each other: IVA fixes the polarity of
each `(subject, component)` pair only up to a flip, because the sources across
subjects are **dependent, not correlated** — flipping a pattern together with its
source leaves the data it explains unchanged. So a component can be perfectly
recovered in every participant and still arrive with half of them inverted.

That is fatal for anything that *compares or averages* participants: the panels of
a participant-comparison figure mix polarities, and the group-mean map cancels the
part the participants actually share. This step resolves the flip per pair, with
`iva_quality.anchor_signs_reference`: correlate that pair's **topography** with
the reference topography from Step 3 and flip it when the correlation is negative.

It is **one** sign per pair, and the pair's TF map carries the same one. A pattern
and its source belong to a single decomposition — flipping one without the other
describes something the IVA never produced, and the two views could then not be
read pair by pair at all. The topography decides it because `ref_topo` is a single
fixed target every participant can be compared against, and because it is the
axis every score is measured on: applying these signs makes `topo_corr_view`
non-negative throughout, so a panel's annotated `r` describes the map beside it.

The driven-response criterion (`iva_quality.anchor_signs_tf`: the mean over
35-45 Hz inside the first 500 ms) is kept as a **diagnostic**. Where it disagrees
with the applied sign, that pair's topography matches the reference while its
40 Hz response runs the other way — a statement about the pair, not a sign left
unresolved, and the reason a component's group-mean TF map can be weaker than its
group-mean topography. The printout reports how often.

The aligned arrays below (`*_view`) are what every figure from Step 6 on uses —
the maps it draws **and** the correlations it scores them by. A scatter point
stands for a pair's topomap and its source, so scoring it under a polarity no
figure shows would rate a decomposition that was never drawn, and its
across-subject mean would cancel exactly as an unaligned group-mean map does.
Because the flip is fixed on the topography, every point sits in the right-hand
half-plane on x; what a participant disagreeing with the reference costs is read
on the **y** — its source need not follow its topography — and in the
per-component sign counts printed below.

In [ ]:
# Patterns first: apply Step 5b's shared component sign (`iva_components` was
# left untouched there), then rescale each subject's pattern to unit L2 norm —
# `iva_g` fixes the source scale but not the pattern scale, so an unnormalised
# high-gain subject would steer every across-subject mean and saturate the shared
# colour limit of the participant figures. Correlations are scale-invariant.
patterns_oriented = normalize_patterns_per_subject(
    iva_components * sign_per_comp[None, :, None]
)

# The one per-pair flip: the pair's topography against the reference topography.
sign_per_subject = iva_quality.anchor_signs_reference(patterns_oriented, ref_topo)
SIGN_ALIGN_NOTE = iva_quality.SUBJECT_SIGN_ANCHOR

# Diagnostic only — would the driven response come out positive under its own
# criterion? Never applied; compared with the flip that is.
sign_tf_subj, _have_tf_anchor = iva_quality.anchor_signs_tf(
    onset_tf, FREQS, epoch_times, resp_duration_s=RESP_DUR_S
)
if _have_tf_anchor:
    TF_ANCHOR_NAME = (
        f"{ASSR_FREQ - iva_quality.TF_ANCHOR_HALFWIDTH_HZ:.0f}-"
        f"{ASSR_FREQ + iva_quality.TF_ANCHOR_HALFWIDTH_HZ:.0f} Hz, "
        f"0-{RESP_DUR_S * 1000:.0f} ms"
    )
else:
    # No bin in the driven band (a band-sliced run): agreeing with the reference
    # map says nothing about the ASSR, but still gives the applied sign something
    # independent to be compared against.
    sign_tf_subj = np.sign(iva_quality.tf_map_correlation(onset_tf, ref_tf))
    sign_tf_subj[sign_tf_subj == 0] = 1.0
    TF_ANCHOR_NAME = "reference TF-map correlation"

# The aligned views every figure below uses — one sign, every domain. The
# correlations come along so a panel's annotated r describes the map beside it,
# and so a scatter point scores the pair the figures actually draw. A source
# flips with its pattern, so the onset-averaged time course and the boxcar
# correlation measured on it carry the same one sign.
patterns_view = patterns_oriented * sign_per_subject[:, :, None]
topo_corr_view = topo_corr * sign_per_subject
onset_tf_view = onset_tf * sign_per_subject[:, :, None, None]
tf_corr_view = tf_corr * sign_per_subject
tf_corr_bands_view = {
    tag: arr * sign_per_subject for tag, arr in tf_corr_bands.items()
}
onset_avg_pca_view = onset_avg_pca * sign_per_subject[:, :, None]
onset_avg_40_view = onset_avg_40 * sign_per_subject[:, :, None]
time_corr_pca_view = time_corr_pca * sign_per_subject
time_corr_40_view = time_corr_40 * sign_per_subject

_n_pairs = n_subjects * N_COMPONENTS_PCA
print(f"Sign anchor    : {SIGN_ALIGN_NOTE}  ->  flipped "
      f"{int((sign_per_subject < 0).sum())}/{_n_pairs} (subject, component) pairs")
print(f"Driven response: {TF_ANCHOR_NAME}  ->  reads positive under that sign for "
      f"{np.mean(sign_per_subject == sign_tf_subj) * 100:.0f}% of pairs")
print("\nPer-component counts (flipped / driven response negative):")
# Every component, not `comp_indices` — that selection is made in Step 6.
for k in range(N_COMPONENTS_PCA):
    _disagree = int((sign_per_subject[:, k] != sign_tf_subj[:, k]).sum())
    print(f"  IC {k + 1:>2}: {int((sign_per_subject[:, k] < 0).sum())} / "
          f"{_disagree}  of {n_subjects} participants")

## Step 6 — Quality scatterplots: the topography x vs the boxcar y

Each point is one `(subject, component)` pair: **x** = topomap correlation with
the reference topography (Step 4), **y** = onset-locked rigid-boxcar correlation
(Step 5). Both are the Step 5c **aligned** scores (`topo_corr_view`,
`time_corr_*_view`): a pair is scored under the one sign its maps are drawn
with, so a dot and the panels behind it describe the same decomposition and a
component's score cannot cancel itself over mixed polarities. For each frequency-reduction variant two figures are produced:

- **Combined** — all points overlaid, **colour = component**, no labels.
- **Per-component** — one small panel per component, points are the subjects
  **labelled by 3-digit ID**; each panel keeps that component's colour so a
  point can be located across both figures.

Both axes are **auto-scaled** to a symmetric window around zero, sized from the
largest correlation magnitude in the figure (plus a little padding) and clamped
to `[-1, 1]`. The same window is shared by the combined and per-component views.

Each component carries a **score** = the mean over subjects of
`(x r + y r) / 2`. It equals `1.0` only when every subject sits at the ideal
**(1, 1)** corner, so a higher score means the component's dots cluster closer to
top-right. The score is shown next to each component in the combined legend and
in each per-component panel title.

The plotting helpers below take the y-axis array explicitly, so **Step 6b reuses
them unchanged** for the TF-map y-axes — only the y (and the label) changes, the
x-axis being the same reference topography throughout. That is what makes the
figures comparable: swapping one axis at a time separates the spatial question
(does the pattern match the reference topography?) from the temporal one (does the
response fill the 500 ms window, or match the reference's whole time-frequency
structure?).

In [ ]:
# Symmetric-window padding: fraction of |r|_max added on each side (a small
# absolute floor keeps very tight clusters from touching the axes).
AXIS_PAD_FRAC = 0.15
AXIS_PAD_MIN = 0.05

# Axis wording, shared with the headless path (src.visualization.iva_quality_plots)
# so a notebook figure and a cluster figure never label the same score differently.
X_LABEL_TOPO = "Topomap correlation with wavelet-PCA PC1 reference"
Y_LABEL_TIME = "Onset-locked time correlation"
Y_LABEL_TF = "Onset-averaged TF-map correlation with wavelet-PCA PC1"

if COMPONENTS_TO_PLOT is None:
    comp_indices = list(range(N_COMPONENTS_PCA))
else:
    comp_indices = list(COMPONENTS_TO_PLOT)

if len(comp_indices) <= 20:
    _comp_colors = list(plt.colormaps["tab20"].colors)
    comp_colors = {k: _comp_colors[i % 20] for i, k in enumerate(comp_indices)}
else:
    cmap = plt.colormaps["hsv"]
    comp_colors = {
        k: cmap(i / len(comp_indices)) for i, k in enumerate(comp_indices)
    }


# The three helpers below still take the x-axis array explicitly (``topo``): it is
# always ``topo_corr_view`` now, but keeping it a parameter means the score helper
# and the two renderers make no assumption about which pair they are given. Every
# array handed to them carries the Step 5c per-pair sign — a score is an
# across-subject mean, so mixed polarities cancel it just as they cancel a map.
def _axis_limit(topo, y_corr):
    """Symmetric [-lim, lim] window from the largest |r| on either axis."""
    max_abs = float(
        max(
            np.abs(topo[:, comp_indices]).max(),
            np.abs(y_corr[:, comp_indices]).max(),
        )
    )
    return min(1.0, max_abs * (1.0 + AXIS_PAD_FRAC) + AXIS_PAD_MIN)


def _component_score(topo, y_corr, k):
    """Per-component quality score: mean over subjects of (topomap + y) / 2.

    Higher is better; the score equals 1.0 exactly when every subject sits at
    the ideal top-right corner (topomap r = 1, y r = 1), so it measures how
    close a component's dots are to (1, 1). It is relative to whichever
    reference pair is passed in, so the families report different scores.
    """
    return float(np.mean((topo[:, k] + y_corr[:, k]) / 2.0))


def plot_onset_diagnostic(onset_avgs, topo, y_corr, *, variant_name, save_name):
    """Per-component group-mean onset-average + the rigid response window.

    Shows, for every component, the across-subject mean onset-triggered response
    on the epoch time axis, with the fixed response window [0, RESP_DUR_S]
    shaded — so you can see whether the component's onset response looks like a
    stimulus response and how well it fills the scored window.

    Participants are put on a common scale before the mean, so the curve is
    a statement about all of them rather than about the loudest few.
    """
    # One scale per subject — the same rule the headless figures use, so a
    # notebook curve and a cluster curve weight the participants alike.
    group_avg = iva_quality.equalize_subject_influence(
        onset_avgs, comp_indices
    ).mean(axis=0)  # (K, W)
    n_comp = len(comp_indices)
    ncols = min(5, n_comp)
    nrows = int(np.ceil(n_comp / ncols))
    fig, axes = plt.subplots(
        nrows, ncols, figsize=(3.0 * ncols, 2.4 * nrows), squeeze=False, sharex=True
    )
    flat = axes.flatten()
    for panel, k in enumerate(comp_indices):
        ax = flat[panel]
        ax.plot(epoch_times, group_avg[k], color=comp_colors[k], lw=1.4)
        ax.axvline(0.0, color="red", ls="--", lw=0.8)
        ax.axvspan(0.0, RESP_DUR_S, color="0.5", alpha=0.15)
        ax.axhline(0.0, color="gray", ls=":", lw=0.5)
        ax.tick_params(labelsize=6)
        ax.set_title(
            f"IC {k + 1}  (score {_component_score(topo, y_corr, k):+.2f})",
            fontsize=8.5, color="black", fontweight="bold",
        )
    for ax in flat[n_comp:]:
        ax.axis("off")
    fig.supxlabel("Time relative to onset (s)")
    fig.supylabel(
        "Group-mean onset-averaged component response\n"
        "(a.u., equal-weighted participants)"
    )
    fig.suptitle(
        f"Onset-locked response vs {RESP_DUR_S * 1000:.0f} ms stimulus window — "
        f"{variant_name} — {LABEL}",
        fontsize=12,
    )
    fig.tight_layout()
    if SAVE_PLOTS:
        fig.savefig(PLOTS_DIR / save_name, dpi=150, bbox_inches="tight")
    plt.show()
    plt.close("all")


def plot_quality_scatter(
    topo, y_corr, *, variant_name, save_name, xlabel=X_LABEL_TOPO, ylabel=None
):
    """Combined scatter: topomap-corr (x) vs the second score (y), colour = component.

    All components/subjects overlaid, no per-point labels. Both axes share one
    symmetric window sized from the largest correlation magnitude present.
    """
    lim = _axis_limit(topo, y_corr)

    fig, ax = plt.subplots(figsize=(9.0, 7.5))
    for s in range(n_subjects):
        for k in comp_indices:
            ax.scatter(
                topo[s, k], y_corr[s, k], color=comp_colors[k],
                marker="o", s=70, edgecolor="black", linewidth=0.3,
                alpha=0.9, zorder=2,
            )
    ax.axhline(0.0, ls="--", lw=0.6, color="gray")
    ax.axvline(0.0, ls="--", lw=0.6, color="gray")
    ax.set_xlim(-lim, lim)
    ax.set_ylim(-lim, lim)
    ax.set_aspect("equal", adjustable="box")
    ax.set_xlabel(xlabel)
    ax.set_ylabel(ylabel or f"{Y_LABEL_TIME} ({variant_name})")
    ax.set_title(
        f"IVA channel-component quality — {variant_name} — {LABEL}\n"
        f"(colour = component; window |r| ≤ {lim:.2f})"
    )

    # Legend labels carry each component's score (closeness to the (1,1) ideal).
    comp_handles = [
        Line2D([0], [0], marker="o", linestyle="", markerfacecolor=comp_colors[k],
               markeredgecolor="black", markersize=8,
               label=f"IC {k + 1}  ({_component_score(topo, y_corr, k):+.2f})")
        for k in comp_indices
    ]
    leg1 = ax.legend(
        handles=comp_handles, title="Component  (score)", fontsize=7,
        loc="upper left", bbox_to_anchor=(1.01, 1.0),
        ncol=1 + (len(comp_indices) > 15),
    )
    ax.add_artist(leg1)
    fig.tight_layout()
    if SAVE_PLOTS:
        # bbox_extra_artists keeps the out-of-axes legend inside the tight crop.
        fig.savefig(
            PLOTS_DIR / save_name, dpi=150, bbox_inches="tight",
            bbox_extra_artists=(leg1,),
        )
    plt.show()
    plt.close("all")


def plot_quality_scatter_per_component(
    topo, y_corr, *, variant_name, save_name, xlabel=X_LABEL_TOPO, ylabel=None
):
    """One small scatter per component; points are subjects labelled by ID.

    Each panel keeps that component's colour (matching the combined plot) and
    the shared symmetric window, so a point can be located across both figures.
    """
    lim = _axis_limit(topo, y_corr)
    n_comp = len(comp_indices)
    ncols = min(5, n_comp)
    nrows = int(np.ceil(n_comp / ncols))
    fig, axes = plt.subplots(
        nrows, ncols, figsize=(2.9 * ncols, 2.9 * nrows), squeeze=False,
    )
    flat = axes.flatten()
    for panel, k in enumerate(comp_indices):
        ax = flat[panel]
        color = comp_colors[k]
        for s in range(n_subjects):
            x, y = topo[s, k], y_corr[s, k]
            ax.scatter(
                x, y, color=color, marker="o", s=55, edgecolor="black",
                linewidth=0.3, alpha=0.9, zorder=2,
            )
            ax.annotate(
                subject_ids[s], (x, y), textcoords="offset points",
                xytext=(3.0, 2.5), fontsize=6, color="black", zorder=3,
            )
        ax.axhline(0.0, ls="--", lw=0.5, color="gray")
        ax.axvline(0.0, ls="--", lw=0.5, color="gray")
        ax.set_xlim(-lim, lim)
        ax.set_ylim(-lim, lim)
        ax.set_aspect("equal", adjustable="box")
        ax.tick_params(labelsize=6)
        ax.set_title(
            f"IC {k + 1}  (score {_component_score(topo, y_corr, k):+.2f})",
            fontsize=8.5, color="black", fontweight="bold",
        )
    for ax in flat[n_comp:]:
        ax.axis("off")
    fig.supxlabel(xlabel)
    fig.supylabel(ylabel or f"{Y_LABEL_TIME} ({variant_name})")
    fig.suptitle(
        f"Per-component quality (label = participant ID) — {variant_name} — {LABEL}"
        f"  |  window |r| ≤ {lim:.2f}",
        fontsize=12,
    )
    fig.tight_layout()
    if SAVE_PLOTS:
        fig.savefig(PLOTS_DIR / save_name, dpi=150, bbox_inches="tight")
    plt.show()
    plt.close("all")


# The Step 5c aligned arrays throughout: the curve is an across-participant mean
# and the score an across-participant mean of correlations, so both cancel under
# the mixed per-pair polarities IVA hands over.
plot_onset_diagnostic(
    onset_avg_pca_view, topo_corr_view, time_corr_pca_view,
    variant_name="PCA frequency-reduction",
    save_name="quality_onset_response_pca.png",
)
plot_quality_scatter(
    topo_corr_view, time_corr_pca_view,
    variant_name="PCA frequency-reduction",
    save_name="quality_scatter_time_pca.png",
)
plot_quality_scatter_per_component(
    topo_corr_view, time_corr_pca_view,
    variant_name="PCA frequency-reduction",
    save_name="quality_scatter_time_pca_per_component.png",
)

In [ ]:
plot_onset_diagnostic(
    onset_avg_40_view, topo_corr_view, time_corr_40_view,
    variant_name="40 Hz band only",
    save_name="quality_onset_response_40hz.png",
)
plot_quality_scatter(
    topo_corr_view, time_corr_40_view,
    variant_name="40 Hz band only",
    save_name="quality_scatter_time_40hz.png",
)
plot_quality_scatter_per_component(
    topo_corr_view, time_corr_40_view,
    variant_name="40 Hz band only",
    save_name="quality_scatter_time_40hz_per_component.png",
)

## Step 6b — Scatterplots: the topography x vs the TF-map y

The same two views as above with the **y** swapped: instead of the boxcar, the
correlation of the component's onset-averaged `(F, W)` map with the reference's
own PC1 score map. The x-axis, the window, the colours and the per-component score
work exactly as in Step 6, so a component sits in a comparable place across every
scatter — only the y-axis moves.

A **TF-map diagnostic** comes first: the group-mean onset-averaged `(F, W)` map
per component with the reference map beside it, so a high or low `tf_corr` can be
looked at rather than trusted. The participants are equal-weighted before the mean
(`iva_quality.equalize_subject_influence`, one scale per subject), so a panel is a
statement about the whole group and not about its loudest few, while the relative
strength of a subject's components survives. Component panels then share one
colour limit, so a component with no onset-locked structure legitimately reads
flat; the reference keeps its own scale, being a PC1 score map rather than an IVA
source.

Three scatter pairs follow, one per TF-map variant from Step 5: the **whole
map**, then **1-10 Hz** and **30-50 Hz**. They share the x-axis, the maps and the
component signs — only the frequency rows the y-score is measured over change —
so a component whose dot climbs from the whole map to 30-50 Hz matches the
reference *in the steady-state band* and is diluted elsewhere.

Read the y-axes against each other rather than any one alone:

- **high on the TF map and on the boxcar** — the component reproduces the
  reference's spectro-temporal structure *and* follows the stimulus window. This
  is the one you want.
- **high on the TF map, low on the boxcar** — it matches the reference's overall
  structure but not the paradigm's timing: look at the onset-response diagnostic
  to see whether it starts late, runs on past the OFF edge, or never stops.
- **high on the boxcar, low on the TF map** — the timing is right but the
  frequency content is not; the TF diagnostic shows where its power actually sits.
- **high on 30-50 Hz, low over the whole map** — the component is the ASSR and
  little else, which the whole-map score dilutes with every frequency the
  reference is quiet in.

In [ ]:
TF_VARIANT = "wavelet PC1 TF map"

# Diagnostic: the maps tf_corr_view is measured on, plus the reference. Drawn on the
# Step 5c aligned maps — an unaligned group mean cancels whatever the
# participants share. Same per-pair sign as the topographies in Step 6c, so a
# participant's two panels belong to one decomposition.
fig = plot_tf_diagnostic(
    onset_tf_view,
    ref_tf,
    topo_corr_view,
    tf_corr_view,
    FREQS,
    epoch_times,
    comp_indices,
    resp_duration_s=RESP_DUR_S,
    assr_freq=ASSR_FREQ,
    label=LABEL,
    alignment_note=SIGN_ALIGN_NOTE,
    save_path=(PLOTS_DIR / "quality_tf_maps_wavelet_tf.png") if SAVE_PLOTS else None,
)
plt.show()
plt.close("all")

# One scatter pair per TF-map variant: the whole map, then each band. Same x,
# same maps, same signs — only the scored frequency rows differ. The Step 5c
# aligned scores, so each y is measured on the maps the diagnostic just drew.
tf_variants = [(None, tf_corr_view, "wavelet_tf")] + [
    (band_name, tf_corr_bands_view[tag], tag)
    for band_name, _fmin, _fmax, tag in iva_quality.TF_BANDS
    if tag in tf_corr_bands_view
]
for _band_name, _y_corr, _tag in tf_variants:
    _variant = (
        TF_VARIANT if _band_name is None else f"{TF_VARIANT}, {_band_name}"
    )
    _ylabel = Y_LABEL_TF if _band_name is None else f"{Y_LABEL_TF} ({_band_name})"
    plot_quality_scatter(
        topo_corr_view, _y_corr,
        variant_name=_variant,
        save_name=f"quality_scatter_{_tag}.png",
        ylabel=_ylabel,
    )
    plot_quality_scatter_per_component(
        topo_corr_view, _y_corr,
        variant_name=_variant,
        save_name=f"quality_scatter_{_tag}_per_component.png",
        ylabel=_ylabel,
    )

# Side-by-side ranking: which components each y-axis favours, and where they
# disagree. "boxcar score" uses the PCA frequency-reduction, "TF score" the
# whole-map correlation; both share the one topography x.
# The band means come last, so a component's whole-map score can be read against
# the bands it is made of ("TF score" stays the whole-map score).
_band_cols = [(name, _y) for name, _y, _t in tf_variants if name is not None]
print(f"{'IC':>4}  {'boxcar':>8}  {'TF score':>8}  {'topo':>8}  "
      f"{'time PCA':>8}  {'TF map':>8}"
      + "".join(f"  {name:>9}" for name, _ in _band_cols))
for k in comp_indices:
    print(
        f"{k + 1:>4}  "
        f"{_component_score(topo_corr_view, time_corr_pca_view, k):>+8.2f}  "
        f"{_component_score(topo_corr_view, tf_corr_view, k):>+8.2f}  "
        f"{topo_corr_view[:, k].mean():>+8.2f}  "
        f"{time_corr_pca_view[:, k].mean():>+8.2f}  "
        f"{tf_corr_view[:, k].mean():>+8.2f}"
        + "".join(f"  {arr[:, k].mean():>+9.2f}" for _name, arr in _band_cols)
    )
_best_box = max(
    comp_indices,
    key=lambda k: _component_score(topo_corr_view, time_corr_pca_view, k),
)
_best_tf = max(
    comp_indices, key=lambda k: _component_score(topo_corr_view, tf_corr_view, k)
)
print(f"\nBest by the boxcar y: IC {_best_box + 1}   "
      f"Best by the TF-map y: IC {_best_tf + 1}"
      f"{'  — same component' if _best_box == _best_tf else '  — THEY DISAGREE'}")

---
## Step 6c — Group-mean topography per component

The spatial counterpart of the TF diagnostic in Step 6b, and the figure to read
*before* the per-participant breakdown in Step 7: one panel per component holding
the **across-participant mean** channel topography, with the reference topomap in
the last panel. It answers "which components look like the reference at all" on a
single page, so only the interesting ones need opening per participant.

One figure: there is one reference topography, and the maps do not depend on
which y-axis is scored. The panel score uses the TF-map y — the reference pair's
own — so the whole figure comes from one reduction.

Conventions match the TF diagnostic exactly. The participants are equal-weighted
before the mean (`iva_quality.equalize_subject_influence`, one scale per subject),
so a panel is a statement about the whole group rather than about its loudest
few, while the relative strength of a subject's components survives. The
component panels then share one colour limit, so a component whose patterns
disagree across participants legitimately reads flat — the incoherent part
cancels in the mean; the reference keeps its own scale, being a PC1 loading
rather than a pattern. Panel titles carry the component's family score and its
mean `r` against the reference shown beside it.

In [ ]:
# `patterns_view` / `topo_corr_view` come from Step 5c: unit-norm per subject and
# per-participant sign-aligned (the pair's topomap-vs-reference correlation). The plotter equalises the per-participant *range*
# on top of that (`equalize_subject_influence`) — unit L2 norm still leaves a focal
# pattern with a larger peak than a diffuse one, so a subject with focal
# topographies would otherwise dominate the mean.
#
# One figure: there is one reference topography, and the maps do not depend on the
# y-axis. The panel score uses the TF-map y, the reference pair's own.
plot_topomap_diagnostic(
    patterns_view,
    ref_topo,
    topo_corr_view,
    tf_corr_view,
    iva_info,
    n_channels,
    comp_indices,
    variant_name=TF_VARIANT,
    label=LABEL,
    alignment_note=SIGN_ALIGN_NOTE,
    save_path=(PLOTS_DIR / "quality_topomaps.png") if SAVE_PLOTS else None,
)
plt.show()
plt.close("all")

---
## Step 7 — One participant-comparison figure per component

Step 6c showed each component's group-mean topography; this step opens the mean
up. The scatterplots collapse each `(subject, component)` pair to two numbers, so
here are the underlying maps, making a low topomap correlation something that can
be *looked at*: one figure per IVA component holding every participant's channel
topography side by side, followed by the group mean and the reference topography.

Numbering is 1-based and follows the **IVA component order**, matching the
`IC <k+1>` labels of the figures above — not the score ranking. Within a figure
every participant panel shares one colour limit so the panels are directly
comparable, and the suptitle reports that limit together with the mean / min /
max `r` across participants. The patterns are first put on a common
per-participant scale (`iva_quality.equalize_subject_influence`), so no one
participant sets that shared limit and every participant carries the same
weight in the group mean; the group-mean panel is then drawn on its **own**
annotated scale, an across-participant mean being weaker than the individual
patterns by construction — otherwise the panel most worth looking at would be
the only unreadable one. Limits are *not* shared across components, whose
pattern magnitudes differ by construction; the reference keeps its own scale
because it is a PC1 loading rather than a pattern. Panels stay in subject order
so a participant sits in the same grid position for every component.

The patterns are `patterns_view` from Step 5c: the shared per-component sign
(Step 5b left `iva_components` untouched), unit L2 norm per subject, and then the
per-pair flip that makes the topography agree with the reference — without that
last step the panels would mix polarities and the group-mean panel would cancel.
The annotated `r` is aligned the same way, so it describes the map drawn beside
it and is non-negative here, while the same participant's dot on the scatters
keeps Step 5b's convention and can sit on the other side of zero.

In [ ]:
# `patterns_view` / `topo_corr_view` (component sign, unit-norm per subject, then
# the per-pair reference-correlation sign) come from Step 5c, exactly as in
# Step 6c — so the two figures show the same maps and weight the participants
# alike.
topomap_root = PLOTS_DIR / "participant_topomaps"
if SAVE_PLOTS:
    written = plot_participant_topomaps(
        patterns_view,
        ref_topo,
        topo_corr_view,
        iva_info,
        n_channels,
        comp_indices,
        subject_ids,
        label=LABEL,
        root_dir=topomap_root,
        alignment_note=SIGN_ALIGN_NOTE,
    )
    print(f"Wrote {len(written)} figures ({len(subject_ids)} participants each).")
    print(f"Directory : {topomap_root}")
    for path in written[:3]:
        print(f"  {path.name}")
    if len(written) > 3:
        print(f"  ... and {len(written) - 3} more")
else:
    print("SAVE_PLOTS is False -> participant-comparison figures skipped.")

---
## Step 7b — One participant-comparison TF-map figure per component

The time-frequency counterpart of Step 7, and the same idea: the Step 6b/6c
scatters collapse each `(subject, component)` pair to a number, so this writes the
underlying **maps** — one figure per IVA component holding every participant's
onset-averaged `(F, W)` map, then the group mean and the wavelet PC1 reference.
A participant dragging a component's `tf_corr` down can be identified here and
looked at, rather than inferred from its dot.

Conventions match Step 7 exactly — including the sign, which is the very same
per-pair flip — so the two directories can be read side by side for the same
component and participant: 1-based numbering in **IVA component
order**, panels in participant order, one shared colour limit per figure (not
across components), participants equal-weighted by
`iva_quality.equalize_subject_influence` so none of them dominates that limit
or the group mean, an own-scale group-mean panel, and the reference on its own
scale because it is a PC1 score map rather than an IVA source. The maps are
`onset_tf_view` from Step 5c — component sign plus the per-pair flip its
topography fixed — which is the same flip the scatters score under, so a
participant's map here and its dot in Step 6b describe one decomposition.
Because that sign is the topography's, a panel may legitimately show a negative
driven response: it is the same flip the participant's topomap panel carries, it
puts the pair below the x-axis on the TF scatters, and Step 5c printed how often
the two criteria part ways.

In [ ]:
tf_root = PLOTS_DIR / "participant_tf_maps"
if SAVE_PLOTS:
    written_tf = plot_participant_tf_maps(
        onset_tf_view,
        ref_tf,
        tf_corr_view,
        FREQS,
        epoch_times,
        comp_indices,
        subject_ids,
        resp_duration_s=RESP_DUR_S,
        assr_freq=ASSR_FREQ,
        label=LABEL,
        root_dir=tf_root,
        alignment_note=SIGN_ALIGN_NOTE,
    )
    print(f"Wrote {len(written_tf)} figures ({len(subject_ids)} participants each).")
    print(f"Directory : {tf_root}")
    for path in written_tf[:3]:
        print(f"  {path.name}")
    if len(written_tf) > 3:
        print(f"  ... and {len(written_tf) - 3} more")
else:
    print("SAVE_PLOTS is False -> participant TF-map figures skipped.")

---
## Step 7c — The whole recording under the same signs (QC)

Everything above is cut on the onset epoch. This step draws the same components
over the **entire recording**, group-averaged under the same per-pair signs from
Step 5c, and it answers two things the epoch cannot: what a component does
*between* the stimuli — drift, a burst of muscle, an electrode going bad — and
whether its onset-locked structure runs through the whole recording or is carried
by one stretch of it.

Read it against the standard IVA time-frequency figure in
[`wavelet_iva_channel.ipynb`](wavelet_iva_channel.ipynb), which averages the raw
`iva_sources`. That average leaves IVA's arbitrary per-`(subject, component)`
polarity in place, so it cancels whatever the participants share: a component can
look empty there and structured here. The printout below quantifies exactly that
— the same equal-weighted mean with and without the signs — so the difference
between the two figures is a number rather than an impression.

Participants are weighted equally here as everywhere else
(`iva_quality.full_tf_group_mean` folds the per-subject scale in, measured one
subject at a time because the full sources are the largest array in the
notebook), and the component panels share one colour limit.

In [ ]:
# The whole recording, not the onset epoch: same components, same per-pair signs
# (Step 5c), every participant weighted equally. `iva_sources` was left untouched
# by Step 5b, so the component orientation is folded into the flip here.
full_tf_mean = iva_quality.full_tf_group_mean(
    iva_sources, sign_per_subject * sign_per_comp[None, :]
)
full_times = np.arange(n_times) / sfreq

plot_full_tf_maps(
    full_tf_mean,
    FREQS,
    full_times,
    comp_indices,
    assr_freq=ASSR_FREQ,
    label=LABEL,
    onset_times=onsets_in / sfreq,
    alignment_note=SIGN_ALIGN_NOTE,
    save_path=(
        (PLOTS_DIR / "quality_tf_maps_full_recording.png") if SAVE_PLOTS else None
    ),
)
plt.show()
plt.close("all")

# The same equal-weighted mean *without* the per-pair signs — what the standard
# IVA time-frequency figure shows. The ratio is how much of that figure was
# cancellation: near 1 the participants already agreed, well above 1 they did not.
_unsigned_mean = iva_quality.full_tf_group_mean(
    iva_sources, np.ones_like(sign_per_subject)
)
print(f"{'IC':>4}  {'|max| signed':>13}  {'|max| unsigned':>15}  {'ratio':>7}")
for k in comp_indices:
    _signed = float(np.abs(full_tf_mean[k]).max())
    _unsigned = float(np.abs(_unsigned_mean[k]).max())
    _ratio = _signed / _unsigned if _unsigned > 0 else float("nan")
    print(f"{k + 1:>4}  {_signed:>13.3g}  {_unsigned:>15.3g}  {_ratio:>6.1f}x")